In [17]:
# Cell 1: Setup and imports
!pip install pytest pytest-cov

import pickle
import hashlib
import random
import string
import math
import sys
import os

print(f"Python version: {sys.version}")
print(f"Pickle version: {pickle.format_version}")
print("Setup complete!")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pickle version: 4.0
Setup complete!


In [18]:
# Cell 2: Helper functions
def pickle_and_hash(obj, protocol=None):
    """
    Pickle an object and return its SHA256 hash.
    If protocol is None, uses default protocol.
    """
    if protocol is None:
        pickled = pickle.dumps(obj)
    else:
        pickled = pickle.dumps(obj, protocol=protocol)
    return hashlib.sha256(pickled).hexdigest()

def test_pickle_determinism(obj, protocol=None, test_name="Test"):
    """
    Generic test function: pickles an object twice and compares hashes.
    Returns True if deterministic, False otherwise.
    """
    hash1 = pickle_and_hash(obj, protocol)
    hash2 = pickle_and_hash(obj, protocol)

    if hash1 == hash2:
        print(f"✅ {test_name} PASSED - Deterministic")
        return True
    else:
        print(f"❌ {test_name} FAILED - Not Deterministic!")
        print(f"   Hash1: {hash1}")
        print(f"   Hash2: {hash2}")
        return False

def run_test_suite(tests):
    """Run a list of test functions"""
    passed = 0
    failed = 0

    for test_func in tests:
        try:
            result = test_func()
            if result:
                passed += 1
            else:
                failed += 1
        except Exception as e:
            print(f"❌ Test {test_func.__name__} crashed with error: {e}")
            failed += 1

    print("\n" + "="*50)
    print(f"Test Suite Results: {passed} passed, {failed} failed")
    print("="*50)

    return passed, failed

In [19]:
# Cell 3: Basic data type tests
def test_integers():
    print("\n--- Testing Integers ---")
    results = []

    integers = [0, 1, -1, 42, 1000, -999, 2**31 - 1, 2**63 - 1]

    for num in integers:
        result = test_pickle_determinism(num, test_name=f"Integer: {num}")
        results.append(result)

    return all(results)

def test_strings():
    print("\n--- Testing Strings ---")
    results = []

    strings = [
        "",
        "Hello",
        "Hello, World!",
        "12345",
        "Special chars: !@#$%^&*()",
        "Unicode: 你好, नमस्ते, مرحبا",
        "a" * 1000,  # Long string
    ]

    for s in strings:
        result = test_pickle_determinism(s, test_name=f"String: {s[:20]}...")
        results.append(result)

    return all(results)

def test_lists():
    print("\n--- Testing Lists ---")
    results = []

    lists = [
        [],
        [1, 2, 3],
        [1, "two", 3.0, [4, 5]],
        list(range(100)),
        [None, True, False, 42, "hello"],
    ]

    for lst in lists:
        result = test_pickle_determinism(lst, test_name=f"List: {str(lst)[:30]}...")
        results.append(result)

    return all(results)

def test_dictionaries():
    print("\n--- Testing Dictionaries ---")
    results = []

    dicts = [
        {},
        {"a": 1, "b": 2},
        {"a": 1, "b": [2, 3], "c": {"d": 4}},
        {1: "one", 2: "two", 3: "three"},
    ]

    for d in dicts:
        result = test_pickle_determinism(d, test_name=f"Dict: {str(d)[:30]}...")
        results.append(result)

    return all(results)

# Run basic tests
basic_tests = [test_integers, test_strings, test_lists, test_dictionaries]
run_test_suite(basic_tests)


--- Testing Integers ---
✅ Integer: 0 PASSED - Deterministic
✅ Integer: 1 PASSED - Deterministic
✅ Integer: -1 PASSED - Deterministic
✅ Integer: 42 PASSED - Deterministic
✅ Integer: 1000 PASSED - Deterministic
✅ Integer: -999 PASSED - Deterministic
✅ Integer: 2147483647 PASSED - Deterministic
✅ Integer: 9223372036854775807 PASSED - Deterministic

--- Testing Strings ---
✅ String: ... PASSED - Deterministic
✅ String: Hello... PASSED - Deterministic
✅ String: Hello, World!... PASSED - Deterministic
✅ String: 12345... PASSED - Deterministic
✅ String: Special chars: !@#$%... PASSED - Deterministic
✅ String: Unicode: 你好, नमस्ते,... PASSED - Deterministic
✅ String: aaaaaaaaaaaaaaaaaaaa... PASSED - Deterministic

--- Testing Lists ---
✅ List: []... PASSED - Deterministic
✅ List: [1, 2, 3]... PASSED - Deterministic
✅ List: [1, 'two', 3.0, [4, 5]]... PASSED - Deterministic
✅ List: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,... PASSED - Deterministic
✅ List: [None, True, False, 42, 'hello... PASSED - Determ

(4, 0)

In [20]:
# Cell 4: Protocol tests
def test_protocols():
    print("\n--- Testing Different Protocols ---")
    results = []

    # Test object
    obj = {"key1": "value1", "key2": [1, 2, 3], "key3": 42.5}

    # All pickle protocols
    protocols = [0, 1, 2, 3, 4, 5]

    print(f"Testing object: {obj}")

    # First check: Each protocol should be deterministic individually
    for protocol in protocols:
        try:
            hash1 = pickle_and_hash(obj, protocol)
            hash2 = pickle_and_hash(obj, protocol)

            if hash1 == hash2:
                print(f"✅ Protocol {protocol} is deterministic")
                results.append(True)
            else:
                print(f"❌ Protocol {protocol} is NOT deterministic!")
                results.append(False)
        except Exception as e:
            print(f"❌ Protocol {protocol} error: {e}")
            results.append(False)

    # Second check: Different protocols should produce different outputs
    print("\n--- Checking Protocol Differences ---")
    hashes = {}
    for protocol in protocols:
        try:
            hashes[protocol] = pickle_and_hash(obj, protocol)
        except:
            hashes[protocol] = None

    # Check if any two protocols produce the same hash (they shouldn't)
    protocol_list = list(hashes.keys())
    for i in range(len(protocol_list)):
        for j in range(i+1, len(protocol_list)):
            p1 = protocol_list[i]
            p2 = protocol_list[j]
            if hashes[p1] is not None and hashes[p2] is not None:
                if hashes[p1] == hashes[p2]:
                    print(f"⚠️  Protocol {p1} and {p2} produced the SAME output!")
                    print(f"   This is unexpected but not necessarily a bug")

    return all(results)

# Run protocol tests
run_test_suite([test_protocols])


--- Testing Different Protocols ---
Testing object: {'key1': 'value1', 'key2': [1, 2, 3], 'key3': 42.5}
✅ Protocol 0 is deterministic
✅ Protocol 1 is deterministic
✅ Protocol 2 is deterministic
✅ Protocol 3 is deterministic
✅ Protocol 4 is deterministic
✅ Protocol 5 is deterministic

--- Checking Protocol Differences ---

Test Suite Results: 1 passed, 0 failed


(1, 0)

In [21]:
# Cell 5: Floating point tests
def test_floating_points():
    print("\n--- Testing Floating Point Numbers ---")
    results = []

    floats = [
        0.0,
        -0.0,
        1.0,
        -1.0,
        3.14159,
        2.71828,
        float('inf'),
        float('-inf'),
        float('nan'),
        math.pi,
        math.e,
        1.23456789012345678901234567890,  # High precision
    ]

    for f in floats:
        result = test_pickle_determinism(f, test_name=f"Float: {f}")
        results.append(result)

    return all(results)

def test_floating_point_accuracy():
    """Test that floats maintain precision"""
    print("\n--- Testing Floating Point Accuracy ---")

    # Test with very close numbers
    numbers = [0.1 + 0.2, 0.3]  # 0.1 + 0.2 might be 0.30000000000000004

    for num in numbers:
        pickled = pickle.dumps(num)
        unpickled = pickle.loads(pickled)

        # Check if they're exactly equal
        if num == unpickled:
            print(f"✅ {num} maintained exact precision")
        else:
            print(f"⚠️  {num} became {unpickled} (precision loss)")

    return True

# Run float tests
run_test_suite([test_floating_points, test_floating_point_accuracy])


--- Testing Floating Point Numbers ---
✅ Float: 0.0 PASSED - Deterministic
✅ Float: -0.0 PASSED - Deterministic
✅ Float: 1.0 PASSED - Deterministic
✅ Float: -1.0 PASSED - Deterministic
✅ Float: 3.14159 PASSED - Deterministic
✅ Float: 2.71828 PASSED - Deterministic
✅ Float: inf PASSED - Deterministic
✅ Float: -inf PASSED - Deterministic
✅ Float: nan PASSED - Deterministic
✅ Float: 3.141592653589793 PASSED - Deterministic
✅ Float: 2.718281828459045 PASSED - Deterministic
✅ Float: 1.2345678901234567 PASSED - Deterministic

--- Testing Floating Point Accuracy ---
✅ 0.30000000000000004 maintained exact precision
✅ 0.3 maintained exact precision

Test Suite Results: 2 passed, 0 failed


(2, 0)

In [22]:
# Cell 6: Complex and recursive structures (CORRECTED VERSION)

# Define classes at the TOP LEVEL (outside any function)
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age
        self.children = []

    def __repr__(self):
        return f"Person({self.name}, {self.age})"

class Calculator:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def add(self):
        return self.x + self.y

def test_nested_structures():
    print("\n--- Testing Nested Structures ---")
    results = []

    nested = {
        "level1": {
            "level2": {
                "level3": {
                    "level4": [1, 2, 3, 4, 5]
                }
            }
        }
    }

    result = test_pickle_determinism(nested, test_name="Deeply nested dict")
    results.append(result)

    # Very deep list
    deep_list = [1]
    for i in range(10):
        deep_list = [deep_list]

    result = test_pickle_determinism(deep_list, test_name="Deeply nested list")
    results.append(result)

    return all(results)

def test_recursive_structures():
    print("\n--- Testing Recursive Structures ---")
    results = []

    # List containing itself
    try:
        recursive_list = []
        recursive_list.append(recursive_list)
        result = test_pickle_determinism(recursive_list, test_name="Self-referential list")
        results.append(result)
    except Exception as e:
        print(f"⚠️  Recursive list caused: {e}")
        results.append(False)

    # Dict containing itself
    try:
        recursive_dict = {}
        recursive_dict['self'] = recursive_dict
        result = test_pickle_determinism(recursive_dict, test_name="Self-referential dict")
        results.append(result)
    except Exception as e:
        print(f"⚠️  Recursive dict caused: {e}")
        results.append(False)

    return all(results)

def test_custom_classes():
    print("\n--- Testing Custom Classes ---")
    results = []

    # Create two identical people
    alice1 = Person("Alice", 30)
    alice2 = Person("Alice", 30)

    # Test same instance
    result = test_pickle_determinism(alice1, test_name="Same instance")
    results.append(result)

    # Test different instances with same data
    result = test_pickle_determinism(alice2, test_name="Different instance same data")
    results.append(result)

    # Test with methods
    calc = Calculator(10, 20)
    result = test_pickle_determinism(calc, test_name="Class with methods")
    results.append(result)

    return all(results)

# Run complex tests
complex_tests = [test_nested_structures, test_recursive_structures, test_custom_classes]
run_test_suite(complex_tests)


--- Testing Nested Structures ---
✅ Deeply nested dict PASSED - Deterministic
✅ Deeply nested list PASSED - Deterministic

--- Testing Recursive Structures ---
✅ Self-referential list PASSED - Deterministic
✅ Self-referential dict PASSED - Deterministic

--- Testing Custom Classes ---
✅ Same instance PASSED - Deterministic
✅ Different instance same data PASSED - Deterministic
✅ Class with methods PASSED - Deterministic

Test Suite Results: 3 passed, 0 failed


(3, 0)

In [23]:
# Cell 7: Fuzzing tests
def generate_random_object(depth=0, max_depth=3):
    """Generate a random Python object for fuzzing"""
    if depth > max_depth:
        return random.randint(0, 100)

    obj_type = random.choice(['int', 'float', 'str', 'list', 'dict', 'bool', 'none', 'tuple', 'set'])

    try:
        if obj_type == 'int':
            return random.randint(-1000, 1000)
        elif obj_type == 'float':
            return random.uniform(-1000, 1000)
        elif obj_type == 'str':
            length = random.randint(1, 20)
            return ''.join(random.choices(string.ascii_letters + string.digits + "!@#$%^&*()", k=length))
        elif obj_type == 'list':
            size = random.randint(0, 5)
            return [generate_random_object(depth + 1, max_depth) for _ in range(size)]
        elif obj_type == 'dict':
            size = random.randint(0, 3)
            d = {}
            for _ in range(size):
                key = generate_random_object(depth + 1, max_depth)
                # Make sure key is hashable (not a dict or list)
                while isinstance(key, (dict, list, set)):
                    key = generate_random_object(depth + 1, max_depth)
                d[key] = generate_random_object(depth + 1, max_depth)
            return d
        elif obj_type == 'bool':
            return random.choice([True, False])
        elif obj_type == 'tuple':
            size = random.randint(0, 4)
            return tuple(generate_random_object(depth + 1, max_depth) for _ in range(size))
        elif obj_type == 'set':
            size = random.randint(0, 3)
            s = set()
            for _ in range(size):
                elem = generate_random_object(depth + 1, max_depth)
                # Make sure element is hashable
                while isinstance(elem, (dict, list, set)):
                    elem = generate_random_object(depth + 1, max_depth)
                s.add(elem)
            return s
        else:  # none
            return None
    except:
        return None

def test_fuzzing():
    print("\n--- Fuzzing Tests (Random Objects) ---")
    results = []

    num_tests = 50  # Start with 50 random objects

    for i in range(num_tests):
        obj = generate_random_object()

        try:
            hash1 = pickle_and_hash(obj)
            hash2 = pickle_and_hash(obj)

            if hash1 == hash2:
                results.append(True)
            else:
                print(f"❌ Fuzzing failed on object: {obj}")
                results.append(False)
        except Exception as e:
            print(f"⚠️  Object {obj} caused error: {e}")
            results.append(False)

    print(f"\nFuzzing results: {sum(results)}/{len(results)} passed")
    return all(results)

def test_fuzzing_with_protocols():
    print("\n--- Fuzzing with Different Protocols ---")
    results = []

    protocols = [0, 1, 2, 3, 4]
    num_tests = 20

    for protocol in protocols:
        protocol_passed = 0
        for i in range(num_tests):
            obj = generate_random_object()
            try:
                hash1 = pickle_and_hash(obj, protocol)
                hash2 = pickle_and_hash(obj, protocol)

                if hash1 == hash2:
                    protocol_passed += 1
                else:
                    print(f"❌ Protocol {protocol} failed on: {obj}")
            except:
                # Some objects might not be pickleable with older protocols
                pass

        success_rate = protocol_passed / num_tests * 100
        print(f"Protocol {protocol}: {protocol_passed}/{num_tests} passed ({success_rate:.1f}%)")
        results.append(protocol_passed == num_tests)

    return all(results)

# Run fuzzing tests
run_test_suite([test_fuzzing, test_fuzzing_with_protocols])


--- Fuzzing Tests (Random Objects) ---

Fuzzing results: 50/50 passed

--- Fuzzing with Different Protocols ---
Protocol 0: 20/20 passed (100.0%)
Protocol 1: 20/20 passed (100.0%)
Protocol 2: 20/20 passed (100.0%)
Protocol 3: 20/20 passed (100.0%)
Protocol 4: 20/20 passed (100.0%)

Test Suite Results: 2 passed, 0 failed


(2, 0)

In [24]:
# Cell 8: Run complete test suite
def run_complete_suite():
    print("="*60)
    print("COMPLETE PICKLE DETERMINISM TEST SUITE")
    print("="*60)
    print(f"Python Version: {sys.version}")
    print(f"Pickle Format Version: {pickle.format_version}")
    print("="*60)

    all_tests = [
        test_integers,
        test_strings,
        test_lists,
        test_dictionaries,
        test_protocols,
        test_floating_points,
        test_floating_point_accuracy,
        test_nested_structures,
        test_recursive_structures,
        test_custom_classes,
        test_fuzzing,
        test_fuzzing_with_protocols
    ]

    return run_test_suite(all_tests)

# Run everything!
run_complete_suite()

COMPLETE PICKLE DETERMINISM TEST SUITE
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pickle Format Version: 4.0

--- Testing Integers ---
✅ Integer: 0 PASSED - Deterministic
✅ Integer: 1 PASSED - Deterministic
✅ Integer: -1 PASSED - Deterministic
✅ Integer: 42 PASSED - Deterministic
✅ Integer: 1000 PASSED - Deterministic
✅ Integer: -999 PASSED - Deterministic
✅ Integer: 2147483647 PASSED - Deterministic
✅ Integer: 9223372036854775807 PASSED - Deterministic

--- Testing Strings ---
✅ String: ... PASSED - Deterministic
✅ String: Hello... PASSED - Deterministic
✅ String: Hello, World!... PASSED - Deterministic
✅ String: 12345... PASSED - Deterministic
✅ String: Special chars: !@#$%... PASSED - Deterministic
✅ String: Unicode: 你好, नमस्ते,... PASSED - Deterministic
✅ String: aaaaaaaaaaaaaaaaaaaa... PASSED - Deterministic

--- Testing Lists ---
✅ List: []... PASSED - Deterministic
✅ List: [1, 2, 3]... PASSED - Deterministic
✅ List: [1, 'two', 3.0, [4, 5]]... PASSED - Det

(12, 0)

In [25]:
# Cell 9: Generate a summary report
def generate_report():
    print("\n" + "="*60)
    print("TEST SUMMARY REPORT")
    print("="*60)

    # Count test functions
    all_test_functions = [
        test_integers, test_strings, test_lists, test_dictionaries,
        test_protocols, test_floating_points, test_floating_point_accuracy,
        test_nested_structures, test_recursive_structures, test_custom_classes,
        test_fuzzing, test_fuzzing_with_protocols
    ]

    print(f"Total test groups: {len(all_test_functions)}")
    print(f"Pickle protocols tested: {[0,1,2,3,4,5]}")
    print(f"Fuzzing iterations: 50 random objects")

    print("\nKey Findings:")
    print("1. Basic data types (int, str, list, dict) are deterministic")
    print("2. All pickle protocols are self-consistent")
    print("3. Different protocols produce different outputs (expected)")
    print("4. Recursive structures work with pickle")
    print("5. Floating point numbers are preserved")

    print("\nLimitations of this test suite:")
    print("- Tested on a single Python version")
    print("- Fuzzing only generates simple objects")
    print("- No cross-platform testing")
    print("- No performance or memory testing")
    print("="*60)

generate_report()


TEST SUMMARY REPORT
Total test groups: 12
Pickle protocols tested: [0, 1, 2, 3, 4, 5]
Fuzzing iterations: 50 random objects

Key Findings:
1. Basic data types (int, str, list, dict) are deterministic
2. All pickle protocols are self-consistent
3. Different protocols produce different outputs (expected)
4. Recursive structures work with pickle
5. Floating point numbers are preserved

Limitations of this test suite:
- Tested on a single Python version
- Fuzzing only generates simple objects
- No cross-platform testing
- No performance or memory testing


In [26]:
# Cell 10: Python version testing
import sys

def test_python_version_compatibility():
    print("\n--- Testing Python Version Compatibility ---")
    print(f"Current Python version: {sys.version}")

    # Test objects
    test_obj = {"a": 1, "b": "test", "c": [1, 2, 3]}

    for protocol in [0, 1, 2, 3, 4, 5]:
        try:
            pickled = pickle.dumps(test_obj, protocol=protocol)
            unpickled = pickle.loads(pickled)

            # Check if round-trip works
            if unpickled == test_obj:
                print(f"✅ Protocol {protocol}: Round-trip successful")
            else:
                print(f"❌ Protocol {protocol}: Round-trip failed")
        except Exception as e:
            print(f"⚠️ Protocol {protocol}: Error - {e}")

    return True

# Run it
test_python_version_compatibility()


--- Testing Python Version Compatibility ---
Current Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
✅ Protocol 0: Round-trip successful
✅ Protocol 1: Round-trip successful
✅ Protocol 2: Round-trip successful
✅ Protocol 3: Round-trip successful
✅ Protocol 4: Round-trip successful
✅ Protocol 5: Round-trip successful


True

In [27]:
# Cell 11: Large object testing
def test_large_objects():
    print("\n--- Testing Large Objects ---")
    results = []

    # Test with progressively larger lists
    sizes = [10, 100, 1000, 10000]

    for size in sizes:
        try:
            large_list = list(range(size))
            result = test_pickle_determinism(
                large_list,
                test_name=f"List of {size} elements"
            )
            results.append(result)
        except Exception as e:
            print(f"⚠️ Size {size} caused error: {e}")
            results.append(False)

    # Test large dictionary
    large_dict = {f"key_{i}": i for i in range(1000)}
    result = test_pickle_determinism(
        large_dict,
        test_name="Dictionary with 1000 keys"
    )
    results.append(result)

    return all(results)

# Run it
test_large_objects()


--- Testing Large Objects ---
✅ List of 10 elements PASSED - Deterministic
✅ List of 100 elements PASSED - Deterministic
✅ List of 1000 elements PASSED - Deterministic
✅ List of 10000 elements PASSED - Deterministic
✅ Dictionary with 1000 keys PASSED - Deterministic


True

In [28]:
# Cell 12: Error handling tests
def test_non_pickleable_objects():
    print("\n--- Testing Non-Pickleable Objects ---")

    # These should fail gracefully (expected behavior)
    non_pickleable = [
        open('/dev/null', 'r'),  # File object
        lambda x: x * 2,          # Lambda function
        __import__('os'),         # Module
    ]

    for obj in non_pickleable:
        try:
            pickle.dumps(obj)
            print(f"⚠️ {type(obj)} was pickled (unexpected!)")
        except (pickle.PicklingError, TypeError, AttributeError) as e:
            print(f"✅ {type(obj)} correctly raised: {type(e).__name__}")

    return True

# Run it
test_non_pickleable_objects()


--- Testing Non-Pickleable Objects ---
✅ <class '_io.TextIOWrapper'> correctly raised: TypeError
✅ <class 'function'> correctly raised: AttributeError
✅ <class 'module'> correctly raised: TypeError


True

In [29]:
# Cell 13: Performance consistency
import time

def test_performance_consistency():
    print("\n--- Testing Performance Consistency ---")

    obj = list(range(10000))
    times = []

    for i in range(5):
        start = time.time()
        pickle.dumps(obj)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    std_dev = (sum((t - avg_time) ** 2 for t in times) / len(times)) ** 0.5

    print(f"Pickling 10,000 elements:")
    print(f"  Average time: {avg_time:.4f} seconds")
    print(f"  Standard deviation: {std_dev:.4f} seconds")
    print(f"  Consistency: {'✅ Good' if std_dev < avg_time * 0.1 else '⚠️ Variable'}")

    return True

# Run it
test_performance_consistency()


--- Testing Performance Consistency ---
Pickling 10,000 elements:
  Average time: 0.0005 seconds
  Standard deviation: 0.0000 seconds
  Consistency: ✅ Good


True